# A/B Test: Cookie Cats
# "Does Moving Progression Gates Affect Player Retention?"

**Business Question:** Cookie Cats, a mobile game, has a specific 'gate' that pauses players to either wait or pay at that level to continue playing. We will run an experiment to determine whether moving the gate from level 30 to leve 40 affects player retention.

**Dataset:** [Mobile Games A/B Testing (Cookie Cats)](https://www.kaggle.com/datasets/mursideyarkin/mobile-games-ab-testing-cookie-cats), ~90,189 players.

**Method:** Randomized A/B test, analyzed with a sample ratio mismatch check, two-proportion
z-tests, and bootstrap confidence intervals.

## 1. Setup

In [7]:
import pandas as pd
import numpy as np
from scipy import stats

df = pd.read_csv('cookie_cats.csv')
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


## 2. Exploratory Data Analysis

In [23]:
print("Missing Values:")
print(df.isnull().sum())

print("\nsum_gamerounds summary:")
print(df['sum_gamerounds'].describe())

print("\nRetention Rates by Group:")
print(df.groupby('version')[['retention_1','retention_7']].mean())

Missing Values:
userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

sum_gamerounds summary:
count    90189.000000
mean        51.872457
std        195.050858
min          0.000000
25%          5.000000
50%         16.000000
75%         51.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64

Retention Rates by Group:
         retention_1  retention_7
version                          
gate_30     0.448188     0.190201
gate_40     0.442283     0.182000


Notes:
- No Missing Values
- Metrics: Our Metrics of concern seem to be the version, retention_1 , and retention_7. We want to determine whether the version (gate 30 or gate 40) affects the retention rates of day 1 and day 7.
- Summary of sum_gamerounds shows that it has an extreme outlier. The max is 49,854 rounds whereas the median is just 16 rounds. We will flag this for now and won't remove it because it doesn't affect the version or retention metrics.  
- Retention rates drop significantly from day 1 to day 7 for both versions. Gate 30 shows slightly slightly better retention for both days which can be due to noise. 

## 3. Check Sample Ratio Mismatch (SRM)

Before moving forward with any testing, first step is to check whether the two groups are equally randomnly distributed. In order to check this, we will perform a chi-square goodness-of-fit test which will help us observe the 50/50 split. 

Hypothesis:

H0: Players were truly randomly assigned 50/50 to each group.

H1: Players were not truly randomly assigned 50/50 to each group.


In [37]:
print("Number of players in each version:")
counts = df['version'].value_counts()
print(counts)

n_total = counts.sum()
expected = [n_total/2 , n_total/2]
observed = [counts['gate_30'], counts['gate_40']]

chi_square_test, p_value = stats.chisquare(observed, expected)

print(f"\nChi-square: {chi_square_test:.4f}, p-value: {p_value:.4f}")


Number of players in each version:
gate_40    45489
gate_30    44700
Name: version, dtype: int64

Chi-square: 6.9024, p-value: 0.0086


**Result:** Our p-value (0.0086) is below the standard 0.05 threshold. Normally when p-value<0.05, we would reject the null hypothesis. In our case, the imbalance seems to be under 1% (49.56% vs 50.44%). Additionally, the industry practice is to use strict SRM threshold less than 0.001 0r 0.0001 because large sample sizes make small imbalances seem significant. Therefore, with the stricter threshold (0.0086<0.0001) we accept the null hypothesis: Players were truly randomly assigned 50/50 to each group.

## 4. Hypothesis Testing: Two-Proportion Z-Test

H0: The true retention rate is the same in both groups.

H1: The true retention rate differs between groups.

Since we're comparing two proportions (percentage retained in gate 30 vs gate 40), the right tool is a two-proportion z-test.


zscore = ((sample_mean1-sample_mean2)-(population_mean_diff))/(math.sqrt((population_std1**2/sample_size1)+(population_std2**2/sample_size2)))

pooled_p = (successes_A + successes_B) / (n_A + n_B)
standard_error = sqrt( pooled_p × (1 - pooled_p) × (1/n_A + 1/n_B) )
z = (rate_A - rate_B) / standard_error

In [43]:
print("\nRetention Rates by Group:")
print(df.groupby('version')[['retention_1','retention_7']].mean())



Retention Rates by Group:
         retention_1  retention_7
version                          
gate_30     0.448188     0.190201
gate_40     0.442283     0.182000


In [ ]:
# 1. Calculate sample proportions
p1_hat = x1 / n1
p2_hat = x2 / n2

# 2. Compute the pooled proportion
p_pooled = (x1 + x2) / (n1 + n2)

# 3. Calculate standard error
standard_error = np.sqrt(p_pooled * (1 - p_pooled) * (1/n1 + 1/n2))

# 4. Compute Z-statistic
z_stat = (p1_hat - p2_hat) / standard_error

# 5. Determine two-tailed P-value
p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

In [60]:
def two_proportion_ztest(df, group_col, outcome_col, group_a, group_b):
    a = df[df[group_col] == group_a][outcome_col]
    b = df[df[group_col] == group_b][outcome_col]

    n_a,n_b = len(a), len(b)
    x_a, x_b = a.sum(), b.sum()
    p_a, p_b = x_a / n_a, x_b / n_b

    p_pooled = (x_a + x_b) / (n_a + n_b)
    standard_error = np.sqrt(p_pooled * (1 - p_pooled) * (1/n_a + 1/n_b))
    z_stat = (p_a - p_b) / standard_error
    p_value = 2 * (1 - stats.norm.cdf(abs(z_stat)))

    se_diff = np.sqrt(p_a*(1-p_a)/n_a + p_b*(1-p_b)/n_b)
    diff = p_a - p_b
    ci_low, ci_high = diff - 1.96*se_diff, diff + 1.96*se_diff
    
    return {
        f'{group_a}_rate': round(p_a, 4),
        f'{group_b}_rate': round(p_b, 4),
        'abs_diff': round(diff, 4),
        'relative_lift_%': round(100 * diff / p_b, 2),
        'z_stat': round(z_stat, 3),
        'p_value': round(p_value, 4),
        '95%_CI_of_diff': (round(ci_low, 4), round(ci_high, 4))
    }

print("Day-1 Retention:")
for k, v in two_proportion_ztest(df, 'version', 'retention_1', 'gate_30', 'gate_40').items():
    print(f"  {k}: {v}")

print("\nDay-7 Retention:")
for k, v in two_proportion_ztest(df, 'version', 'retention_7', 'gate_30', 'gate_40').items():
    print(f"  {k}: {v}")

Day-1 Retention:
  gate_30_rate: 0.4482
  gate_40_rate: 0.4423
  abs_diff: 0.0059
  relative_lift_%: 1.34
  z_stat: 1.784
  p_value: 0.0744
  95%_CI_of_diff: (-0.0006, 0.0124)

Day-7 Retention:
  gate_30_rate: 0.1902
  gate_40_rate: 0.182
  abs_diff: 0.0082
  relative_lift_%: 4.51
  z_stat: 3.164
  p_value: 0.0016
  95%_CI_of_diff: (0.0031, 0.0133)


Results:

| Metric | gate_30 | gate_40 | Abs. diff | Relative lift | p-value | 95% CI |
|---|---|---|---|---|---|---|
| Day-1 retention | 44.82% | 44.23% | 0.59pp | 1.34% | 0.0744 | (-0.06pp, 1.24pp) |
| Day-7 retention | 19.02% | 18.20% | 0.82pp | 4.51% | **0.0016** | (0.31pp, 1.33pp) |

- Retention Day 1: Since p-value>0.05 and the confidence interval barely contains 0, the results are not statistically significant.

- Retention Day 7: Since p-value<0.05 and the confidence interval is above 0, the results are statistically significant.

## 5. Recommendation and Conclusion

**Recommendation:** Do not move gate from level 30 to level 40.
    
From our hypotheses testing results, we saw that the retention rate for day 7 was significantly lower in gate 40 with a 95% confidence interval entirely below control's rate. Day 1 retention showed the trend in the same direction but results were not statistically significant. Therefore, the recommendation is to keep the gate at its current position (level 30) because moving it will hurt long-term engagement. 